Neural Training

In [2]:
# 1. Install dependencies first! Run this in a separate cell:
!pip install -q -U torch transformers peft accelerate trl bitsandbytes datasets pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.


In [3]:
!pip uninstall -y pyarrow datasets
!pip install pyarrow==15.0.2 datasets==2.19.2

Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
Found existing installation: datasets 5.0.0
Uninstalling datasets-5.0.0:
  Successfully uninstalled datasets-5.0.0
  Using cached pyarrow-15.0.2-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached datasets-2.19.2-py3-none-any.whl.metadata (19 kB)
Using cached pyarrow-15.0.2-cp312-cp312-manylinux_2_28_x86_64.whl (38.3 MB)
Using cached datasets-2.19.2-py3-none-any.whl (542 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 1.6.0 requires datasets>=4.7.0, but you have datasets 2.19.2 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 2

In [4]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

ModuleNotFoundError: Could not import module 'TrainingArguments'. Are this object's requirements defined correctly?

In [ ]:
print("--- STEP 1: MERGING DATASETS ---")
# Load the cleaned datasets you created earlier
codex_df = pd.read_csv("codex_clean.csv")
mqa_df = pd.read_csv("unified_mqa_train.csv")

In [ ]:
master_train_df = pd.concat([codex_df, mqa_df], ignore_index=True)
master_train_df = master_train_df.sample(frac=1, random_state=42).reset_index(drop=True) # Shuffle
dataset = Dataset.from_pandas(master_train_df)


In [ ]:
model_id = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def format_prompt(row):
    """The strict format we force the AI to learn."""
    prompt = f"### Instruction: Write Python code to solve the math problem. Store the answer in 'result'.\n"
    prompt += f"### Question:\n{row['question']}\n"
    prompt += f"### Code:\n{row['code']}"
    return {"text": prompt + tokenizer.eos_token}

formatted_dataset = dataset.map(format_prompt)

In [ ]:
print("--- STEP 2: LOADING MODEL IN 4-BIT ---")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

In [ ]:
print("--- STEP 3: APPLYING QLoRA ---")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)


In [ ]:
print("--- STEP 4: TRAINING ---")
training_args = TrainingArguments(
    output_dir="./phi3-math-agent",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=500, # Use max_steps for a quick test, or change to num_train_epochs=1 for full training
    fp16=True,
    optim="paged_adamw_8bit"
)


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()


In [ ]:
print("--- STEP 5: SAVING ---")
trainer.model.save_pretrained("phi3-neuro-symbolic-adapter")
tokenizer.save_pretrained("phi3-neuro-symbolic-adapter")
print("✅ DONE! Member 1, zip the 'phi3-neuro-symbolic-adapter' folder and give it to Member 3.")